In [1]:
from mlxtend.frequent_patterns import fpgrowth, association_rules
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_excel('online_retail_II.xlsx')

In [3]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


In [5]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,525461.000000,525461,525461.000000,417534.000000
mean,10.337667,2010-06-28 11:37:36.845017856,4.688834,15360.645478
min,-9600.000000,2009-12-01 07:45:00,-53594.360000,12346.000000
25%,1.000000,2010-03-21 12:20:00,1.250000,13983.000000
50%,3.000000,2010-07-06 09:51:00,2.100000,15311.000000
75%,10.000000,2010-10-15 12:45:00,4.210000,16799.000000
max,19152.000000,2010-12-09 20:01:00,25111.090000,18287.000000
std,107.424110,NaN,146.126914,1680.811316


# Preprocessing

We will drop rows where Customer ID is null since it won't help us in any way, We will also drop rows in Price and Quantitiy where the value is -ve since it might indicate cancellations or returns or not valid values in general. We will drop InvoiceDate and Country too since they are Irrelevant to what we are doing.

In [7]:
#drop cancellations and returns

df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]
df = df.dropna(subset=['Customer ID']) #drop rows with missing CustomerID

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 407664 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      407664 non-null  object        
 1   StockCode    407664 non-null  object        
 2   Description  407664 non-null  object        
 3   Quantity     407664 non-null  int64         
 4   InvoiceDate  407664 non-null  datetime64[ns]
 5   Price        407664 non-null  float64       
 6   Customer ID  407664 non-null  float64       
 7   Country      407664 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 28.0+ MB


In [9]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,407664.000000,407664,407664.000000,407664.000000
mean,13.585585,2010-07-01 10:15:11.871688192,3.294438,15368.592598
min,1.000000,2009-12-01 07:45:00,0.001000,12346.000000
25%,2.000000,2010-03-26 14:01:00,1.250000,13997.000000
50%,5.000000,2010-07-09 15:47:00,1.950000,15321.000000
75%,12.000000,2010-10-14 17:09:00,3.750000,16812.000000
max,19152.000000,2010-12-09 20:01:00,10953.500000,18287.000000
std,96.840747,NaN,34.757965,1679.762138


In [10]:
df = df.drop(['InvoiceDate', 'Country'], axis=1)

In [11]:
df.head()

,Invoice,StockCode,Description,Quantity,Price,Customer ID
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,13085.0
1,489434,79323P,PINK CHERRY LIGHTS,12,6.75,13085.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,6.75,13085.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2.10,13085.0
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,13085.0


In [12]:
#ensure right data types

df['StockCode'] = df['StockCode'].astype(str)
df['Customer ID'] = df['Customer ID'].astype(int)
df['Description'] = df['Description'].astype(str)
df['Invoice'] = df['Invoice'].astype(int)

In [13]:
df.dtypes

Invoice          int32
StockCode       object
Description     object
Quantity         int64
Price          float64
Customer ID      int32
dtype: object

In [14]:
#dropping duplicate rows

df = df.drop_duplicates()

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 400915 entries, 0 to 525460
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      400915 non-null  int32  
 1   StockCode    400915 non-null  object 
 2   Description  400915 non-null  object 
 3   Quantity     400915 non-null  int64  
 4   Price        400915 non-null  float64
 5   Customer ID  400915 non-null  int32  
dtypes: float64(1), int32(2), int64(1), object(2)
memory usage: 18.4+ MB


# Getting Rules

## Association Rules

1. Support — how often do these items appear together across ALL transactions
    - Low support = rare combination, High support = very common combination

2. Confidence — given that the customer bought item A, how likely are they to also buy item B
    - Example: conf = 0.7 means 70% of people who bought A also bought B, It's basically a conditional probability

3. Lift — is this association actually meaningful or just a coincidence?
    - Lift = 1 → pure coincidence, the items are independent
    - Lift > 1 → they actually go together more than random chance
    - Lift < 1 → buying A actually makes B less likely
    - We always want lift > 1, otherwise the rule is useless

## Approach

- We will use Invoice and StockCode for Association Rules as they represent a cart or a basket of products
- We will use StockCode and Descripition for TF-IDF to mine for words in descripition per product

In [17]:
basket = df.groupby(['Invoice', 'StockCode'])['Quantity'].sum().unstack().fillna(0)

#groupby → groups by invoice and product
#sum() → adds up quantities
#unstack() → pivots StockCode into columns
#fillna(0) → fills missing combinations with 0

basket = basket.map(lambda x: 1 if x > 0 else 0).astype(bool)

#Converts all numbers into just 0 or 1 (bought or not bought), because association rules don't care about quantity,
#just whether the item was present, converted to bool as fpgrowth algorithm works better with it

In [18]:
# Start with these

min_support = 0.01
min_confidence = 0.5
min_lift = 1.0

In [19]:
#running fp_growth

frequent_itemsets = fpgrowth(basket, min_support, use_colnames=True)

In [20]:
#generating rules

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_confidence)
rules = rules[rules['lift'] >= min_lift]

In [21]:
print(f"Number of rules: {len(rules)}")
print(rules.head())

Number of rules: 244
  antecedents consequents  antecedent support  consequent support   support  \
0     (22064)     (21232)            0.016707            0.069484  0.010306   
1     (21755)     (21754)            0.049394            0.061104  0.027169   
2     (20971)     (20972)            0.029667            0.037162  0.018061   
3     (22274)     (22271)            0.018685            0.023161  0.010878   
4     (22274)     (22273)            0.018685            0.021756  0.010149   

   confidence       lift  representativity  leverage  conviction  \
0    0.616822   8.877161               1.0  0.009145    2.428419   
1    0.550053   9.001842               1.0  0.024151    2.086679   
2    0.608772  16.381422               1.0  0.016958    2.461065   
3    0.582173  25.135470               1.0  0.010445    2.337900   
4    0.543175  24.966580               1.0  0.009743    2.141400   

   zhangs_metric   jaccard  certainty  kulczynski  
0       0.902429  0.135802   0.588209    0.

# Output Explaination

antecedents: {22064}  →  consequents: {21232}
Means: customers who bought product 22064 also bought product 21232

**The numbers**:

- antecedent support = 0.016707 → Product 22064 appears in 1.67% of all transactions
- consequent support = 0.069484 → Product 21232 appears in 6.94% of all transactions
- support = 0.010306 → Both products appear together in 1.03% of all transactions
- confidence = 0.616822 → 61.7% of customers who bought 22064 also bought 21232
- lift = 8.877 → Customers who bought 22064 are 8.8x more likely to buy 21232 than a random customer. This is a very strong association

**So in Summary**:

If a customer buys product 22064, there's a 61.7% chance they'll also buy product 21232, and this is 8.8 times stronger than random chance

 # TF-IDF Vectors for Products

In this step, the textual descriptions of the products are converted into numerical vectors using the TF-IDF (Term Frequency-Inverse Document Frequency) technique. This allows the system to calculate the similarity between products based on their content rather than just transaction patterns.

A unique mapping of StockCode to Description is created to ensure each product has a single representative vector. The cosine_similarity matrix is then computed, which provides a score between 0 and 1 indicating how similar any two products are based on their descriptions.

In [29]:
# 1. Prepare unique item descriptions and ensure they match the dataset
# We reset the index to ensure our matrix indices align perfectly (0 to N-1)
item_descriptions = df[['StockCode', 'Description']].drop_duplicates(subset='StockCode').reset_index(drop=True)
item_descriptions['Description'] = item_descriptions['Description'].fillna('Unknown')

# 2. Build the TF-IDF Matrix
# stop_words='english' removes common words like 'the', 'a', 'in', etc.
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(item_descriptions['Description'])

# 3. Compute the Cosine Similarity Matrix
# This results in a square matrix where [i, j] represents the similarity between item i and item j
content_sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

# 4. Create lookup mappings for steps 4 and 5
# This maps each StockCode to its row/column index in the content_sim_matrix
item_index_mapping = pd.Series(item_descriptions.index, index=item_descriptions['StockCode']).to_dict()
reverse_index_mapping = {v: k for k, v in item_index_mapping.items()}

print(f"Successfully generated TF-IDF similarity matrix.")
print(f"Matrix shape: {content_sim_matrix.shape} (Aligned with {len(item_descriptions)} unique products)")

Successfully generated TF-IDF similarity matrix.
Matrix shape: (4017, 4017) (Aligned with 4017 unique products)


# Candidate Generation

This step involves identifying "candidate" items for a specific user. The system looks at the user's purchase history (seed items) and pulls potential recommendations from two sources:

Association Rules (AR): Items that are frequently bought alongside the user's seed items (using the rules generated in Step 2).

Content-Based Filtering: Items that have high cosine similarity to the user's seed items based on the TF-IDF vectors from Step 3.

In [33]:
def get_candidates(customer_id, df_transactions, ar_rules, similarity_matrix, top_n_content=10):
    # Get the user's history
    user_history = set(df_transactions[df_transactions['Customer ID'] == customer_id]['StockCode'].unique())
    
    if not user_history:
        return {}, {}

    # 1. Generate AR Candidates
    ar_candidates = {}
    # Filter rules where the antecedent is a subset of the user's history
    matching_rules = ar_rules[ar_rules['antecedents'].apply(lambda x: x.issubset(user_history))]
    
    for _, row in matching_rules.iterrows():
        for item in row['consequents']:
            if item not in user_history:
                # Store the highest confidence score for each candidate
                ar_candidates[item] = max(ar_candidates.get(item, 0), row['confidence'])

    # 2. Generate Content-Based Candidates
    content_candidates = {}
    for item in user_history:
        if item in item_index_mapping:
            idx = item_index_mapping[item]
            # Get similarity scores for this item against all others
            sim_scores = list(enumerate(similarity_matrix[idx]))
            # Sort and take top N similar items
            sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n_content+1]
            
            for sim_idx, score in sim_scores:
                candidate_item = reverse_index_mapping.get(sim_idx)
                if candidate_item and candidate_item not in user_history:
                    content_candidates[candidate_item] = max(content_candidates.get(candidate_item, 0), score)
                    
    return ar_candidates, content_candidates

# Test candidate generation for a sample user
ar_cands, ct_cands = get_candidates(13085, df, rules, content_sim_matrix)
print(f"Generated {len(ar_cands)} AR candidates and {len(ct_cands)} Content candidates.")

Generated 2 AR candidates and 248 Content candidates.


# Hybrid Scoring and Final Ranking

In the final step of the recommendation logic, the candidates from both methods are merged and scored using a weighted hybrid formula:$Hybrid\_Score = \alpha \times (AR\_Confidence) + (1 - \alpha) \times (Content\_Similarity)$
By adjusting the $\alpha$ parameter, the system can prioritize either the collaborative behavior (AR) or the item attributes (Content). The final output is a ranked list of items that the user is most likely to interact with.

In [40]:
def rank_hybrid_recommendations(ar_candidates, content_candidates, alpha=0.5, k=5):
    combined_items = set(ar_candidates.keys()).union(set(content_candidates.keys()))
    final_scores = []

    for item in combined_items:
        ar_score = ar_candidates.get(item, 0)
        content_score = content_candidates.get(item, 0)
        
        # Weighted hybrid formula
        score = (alpha * ar_score) + ((1 - alpha) * content_score)
        final_scores.append({
            'StockCode': item,
            'HybridScore': score,
            'AR_Score': ar_score,
            'Content_Score': content_score
        })

    # Sort by score descending
    ranked_list = sorted(final_scores, key=lambda x: x['HybridScore'], reverse=True)
    return pd.DataFrame(ranked_list).head(k)

# Generate final recommendations for the test user
recommendations = rank_hybrid_recommendations(ar_cands, ct_cands, alpha=0.5, k=5)

# Attach descriptions for readability
recommendations = recommendations.merge(
    item_descriptions[['StockCode', 'Description']], on='StockCode', how='left'
)

print("Top 5 Hybrid Recommendations:")
recommendations[['StockCode', 'Description', 'HybridScore', 'AR_Score', 'Content_Score']]

Top 5 Hybrid Recommendations:


,StockCode,Description,HybridScore,AR_Score,Content_Score
0,21212,PACK OF 72 RETRO SPOT CAKE CASES,0.566239,0.527378,0.605101
1,84991,60 TEATIME FAIRY CAKE CASES,0.510864,0.520173,0.501555
2,22703,PINK CAT BOWL,0.462579,0.000000,0.925157
3,22199,FRYING PAN RED POLKADOT,0.459952,0.000000,0.919904
4,22203,MILK PAN RED RETROSPOT,0.453040,0.000000,0.906079


# Output Explanation

#### Recommendation: 
If a user has interacted with items similar to "PACK OF 72 RETRO SPOT CAKE CASES," the system recommends the "PINK CAT BOWL."

#### The Numbers:
AR_Score = 0.000000: This indicates there is no direct transaction-based rule linking the user's past purchases to the "PINK CAT BOWL."

Content_Score = 0.925157: This shows that the product description of the "PINK CAT BOWL" is 92.5% similar to the user's past purchase history (calculated via TF-IDF cosine similarity).

HybridScore = 0.462579: This is the weighted average (50/50 split) of the AR and Content scores.

#### So in Summary:
Even though the transaction logs (AR) don't suggest this user would buy the "PINK CAT BOWL," the system recommends it because its product attributes strongly match the user's previous preferences. This confirms that the hybrid approach successfully uses content similarity to overcome the "cold start" limitation of association rules.